In [ ]:
import cv2
import pandas
import tensorflow
import numpy
import torchvision
import torchvision.transforms as transforms
import torch
import numpy as np
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from matplotlib import pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import keras


In [ ]:
image_name_iterator = 0;
image_tensors = [];
real_labels = pandas.DataFrame();
row_index = 0;

In [ ]:
def row_to_tensor(row):
    pixels = np.array([]);
    for i in row:
        pixels = np.append(pixels, i);
    image_tensors.append(torch.tensor(pixels));
    return row;
def normalization(df):
    df_min_max_scaled = df.copy() 
  
    for column in df_min_max_scaled.columns: 
        df_min_max_scaled[column] = (df_min_max_scaled[column] - df_min_max_scaled[column].min()) / (df_min_max_scaled[column].max() - df_min_max_scaled[column].min())     
    
    return df_min_max_scaled;





In [ ]:
data = pandas.read_csv("train.csv");

#data = data.head();

display(data);

#data = data.head();
labels = pandas.DataFrame();
labels['l'] = data['label'];
data.drop(columns=['label'], inplace=True);

for i in range(10):
    real_labels.insert(i, str(i), np.zeros(42000));




display(labels);

#display(data);
#data = data.apply(row_to_tensor, axis=1);
#print(image_tensors);
#plt.show();

In [ ]:
display(labels);

In [ ]:
it = 0;

real_labels['0'] = labels['l'].apply(lambda x: 1 if x == 0 else 0);
real_labels['1'] = labels['l'].apply(lambda x: 1 if x == 1 else 0);
real_labels['2'] = labels['l'].apply(lambda x: 1 if x == 2 else 0);
real_labels['3'] = labels['l'].apply(lambda x: 1 if x == 3 else 0);
real_labels['4'] = labels['l'].apply(lambda x: 1 if x == 4 else 0);
real_labels['5'] = labels['l'].apply(lambda x: 1 if x == 5 else 0);
real_labels['6'] = labels['l'].apply(lambda x: 1 if x == 6 else 0);
real_labels['7'] = labels['l'].apply(lambda x: 1 if x == 7 else 0);
real_labels['8'] = labels['l'].apply(lambda x: 1 if x == 8 else 0);
real_labels['9'] = labels['l'].apply(lambda x: 1 if x == 9 else 0);

display(real_labels);

In [ ]:
np.random.seed(0)
random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(0)


In [ ]:
device = torch.device("mps")
print(device)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()


In [ ]:
real_labels_2 = pandas.DataFrame();
labels_2 = pandas.DataFrame();
labels_2['l'] = y_train;

real_labels_2['0'] = labels_2['l'].apply(lambda x: 1 if x == 0 else 0);
real_labels_2['1'] = labels_2['l'].apply(lambda x: 1 if x == 1 else 0);
real_labels_2['2'] = labels_2['l'].apply(lambda x: 1 if x == 2 else 0);
real_labels_2['3'] = labels_2['l'].apply(lambda x: 1 if x == 3 else 0);
real_labels_2['4'] = labels_2['l'].apply(lambda x: 1 if x == 4 else 0);
real_labels_2['5'] = labels_2['l'].apply(lambda x: 1 if x == 5 else 0);
real_labels_2['6'] = labels_2['l'].apply(lambda x: 1 if x == 6 else 0);
real_labels_2['7'] = labels_2['l'].apply(lambda x: 1 if x == 7 else 0);
real_labels_2['8'] = labels_2['l'].apply(lambda x: 1 if x == 8 else 0);
real_labels_2['9'] = labels_2['l'].apply(lambda x: 1 if x == 9 else 0);

In [ ]:
display(labels_2);
display(real_labels_2);

In [ ]:


device = torch.device("mps")
print(device)

# load the dataset, split into input (X) and output (y) variables
dataset = data;


X = data.to_numpy();
y = real_labels.to_numpy();
print(X);
y = np.append(y, real_labels_2.to_numpy());

for it in range(len(x_train)):
    if(it % 100 == 0): print((it / 60000) * 100);
    listt = np.array([], dtype=int);
    for i in range(len(x_train[it])):
        for j in range(len(x_train[it][i])):
            listt = np.append(listt, x_train[it][i][j]);
    #print(listt);
    X = np.append(X, listt);
    if(it == 1000): break;

#print(x_train);
#X = np.append(X, x_train);
#y = np.append(y, y_train);
X = X.reshape((-1, 784))
print(X);


X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y, dtype=torch.float32).to(device)
print(X_tensor);
print(y_tensor);



In [ ]:

# define the model
model = nn.Sequential(
    nn.Linear(784, 784),
    nn.ReLU(),
    #nn.Linear(784, 784),
    #nn.ReLU(),
    nn.Linear(784, 10),
    nn.Softmax()
)
model.to(device);
print(model)

# train the model
loss_fn   = nn.L1Loss()  # binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=0.0000001)

n_epochs = 200
batch_size = 10

for epoch in range(n_epochs):
    for i in range(0, len(X_tensor), batch_size):
        if(i % 8000 == 0): 
            print(i / len(X_tensor) * 100, '% epoch', epoch);
            #for param in model.parameters():
            #    print(param.data)
        Xbatch = X_tensor[i:i+batch_size].to(device)
        #print(Xbatch.shape);
        y_pred = model(Xbatch).to(device);
        ybatch = y_tensor[i:i+batch_size].to(device);
        #print(y_pred);
        #print(y_tensor);
        #print("model.co: ", model.parameters());
        loss = loss_fn(y_pred, ybatch).to(device);
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Finished epoch {epoch}, latest loss {loss}')

# compute accuracy (no_grad is optional)

In [ ]:
with torch.no_grad():
    y_pred = model(X_tensor).to(device);
    y_pred.to(device);
print(y_pred);
print(y_tensor)
accuracy = (y_pred.round() == y)
print(f"Accuracy {accuracy}")

In [ ]:
test = pandas.read_csv("test1.csv");

arr = test.to_numpy();

print(arr);

X_test_tensor = torch.tensor(arr, dtype=torch.float32).to(device);

print(X_test_tensor);

print(model(X_test_tensor))

In [ ]:
pandas.set_option('display.max_columns', None)

mat = model(X_test_tensor).cpu().data.numpy();
print(len(mat));

ans = pandas.DataFrame();

file = open("ansGPU.csv", 'w');

for i in range(len(mat)):
    mx = mat[i].argmax();
    file.write(str(i + 1) + ',' + str(mx) + "\n");
    #print("i: ", i, " mx:", mx)
    
